# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdullah-pharmd/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Signal checks and verdicts
Before building our rule-based baseline, we check our two primary signals in the starter dataset to see if they are associated with actual traffic decline (`trend_direction == 'down'`).

#### Signal 1: Freshness (Staleness)
- **Logic**: Older content (higher `days_since_last_update`) is more likely to decline in search traffic due to staleness.
- **Verdict**: **CONFIRMED** (after applying a minimum volume floor of 100 impressions to remove low-volume noise). Without the floor, the results are distorted (very stale pages look artificially stable). With the floor, the decline rate increases monotonically as pages get older:
  - `0-30` days freshness: **58.27%** decline rate ($n = 13,735$)
  - `31-90` days freshness: **59.21%** decline rate ($n = 152$)
  - `91-180` days freshness: **62.25%** decline rate ($n = 8,084$)
  - `181+` days freshness: **74.29%** decline rate ($n = 35$)

#### Signal 2: Click-Through Rate (CTR) Underperformance
- **Logic**: Pages that capture fewer clicks than expected for their position tier are underperforming and are highly vulnerable to traffic decline.
- **Verdict**: **CONFIRMED** (using median CTR of each position tier as a benchmark). Pages with CTR below their position tier's median have a decline rate of **65.95%** ($n = 10,307$), whereas pages with CTR at or above their tier's median have a decline rate of **54.32%** ($n = 11,699$).

### The Rule and Reason Codes
Our baseline rule aggregates these signals to prioritize pages for review first. We calculate a composite score: 
$$\text{Baseline Score} = 0.40 \times \text{Visibility} + 0.30 \times \text{Freshness Risk} + 0.25 \times \text{Position Opportunity} + 0.05 \times \text{Depth Gap}$$

We assign **one primary reason code** and **suggested action** per page based on its risk profile:
1. `stale_visible_page` -> **Action**: `refresh` (updated $\ge 180$ days ago & GSC impressions $\ge 500$).
2. `low_ctr_visible_page` -> **Action**: `refresh_and_review_ctr` (CTR below 0.5% & GSC impressions $\ge 500$ & rank on page 1-2).
3. `thin_visible_page` -> **Action**: `expand_and_refresh` (word count < 1200 & GSC impressions $\ge 250$).
4. `page_one_decay_risk` -> **Action**: `refresh` (rank on page 1 & page age $\ge 180$ days).
5. `general_refresh_review` -> **Action**: `monitor` (default check).

In [1]:
# Signal check and verification tables in the starter dataset
import pandas as pd
import numpy as np

csv_path = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_path)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# Apply standard volume floor of 100 impressions and valid avg_position
df_filtered = df[(df['avg_position'] > 0) & (df['impressions_90d'] >= 100)].copy()

print("=== Signal 1: Freshness Tier vs. Decline (with Volume Floor) ===")
freshness_check = df_filtered.groupby('freshness_tier').agg(
    total_pages=('is_declining', 'count'),
    declining_pages=('is_declining', 'sum'),
    decline_rate=('is_declining', 'mean')
).reset_index()
print(freshness_check.to_string(index=False))

print("\n=== Signal 2: Position Tier CTR Benchmarks (with Volume Floor) ===")
position_benchmarks = df_filtered.groupby('position_tier').agg(
    median_ctr=('ctr', 'median'),
    count=('ctr', 'count')
).reset_index()
print(position_benchmarks.to_string(index=False))

# Check low CTR vs. decline
df_with_bench = df_filtered.merge(position_benchmarks[['position_tier', 'median_ctr']], on='position_tier', how='left')
df_with_bench['is_low_ctr'] = (df_with_bench['ctr'] < df_with_bench['median_ctr']).astype(int)

print("\n=== Signal 2 Check: Low CTR for Position vs. Decline ===")
ctr_decline_check = df_with_bench.groupby('is_low_ctr').agg(
    total_pages=('is_declining', 'count'),
    declining_pages=('is_declining', 'sum'),
    decline_rate=('is_declining', 'mean')
).reset_index()
print(ctr_decline_check.to_string(index=False))


=== Signal 1: Freshness Tier vs. Decline (with Volume Floor) ===
freshness_tier  total_pages  declining_pages  decline_rate
          0-30        13735             8004      0.582745
          181+           35               26      0.742857
         31-90          152               90      0.592105
        91-180         8084             5032      0.622464

=== Signal 2: Position Tier CTR Benchmarks (with Volume Floor) ===
position_tier  median_ctr  count
         deep        0.00    879
       page_1        0.23   8633
     page_3_5        0.06   6058
     striking        0.15   5903
        top_3        0.19    533

=== Signal 2 Check: Low CTR for Position vs. Decline ===
 is_low_ctr  total_pages  declining_pages  decline_rate
          0        11699             6355      0.543209
          1        10307             6797      0.659455


## 2. Build the ranked queue (writes the CSV)

We calculate the baseline scores, sort them, assign reason codes, and write the output to `work/outputs/baseline_action_score.csv` as our priority queue.

In [2]:
import os

# Reset to full dataset to build queue
df_full = pd.read_csv(csv_path)
df_full['is_declining_label'] = (df_full['trend_direction'] == 'down').astype(int)

def percentile_rank(series):
    return series.rank(pct=True)

def normalize(series):
    s_min = series.min()
    s_max = series.max()
    if s_max == s_min:
        return series * 0
    return (series - s_min) / (s_max - s_min)

# Calculate sub-scores
df_full["visibility_score"] = percentile_rank(np.log1p(df_full["impressions_90d"]))
df_full["freshness_risk_score"] = percentile_rank(df_full["days_since_last_update"])
df_full["position_opportunity_score"] = (
    (1 - normalize(df_full["avg_position"].clip(lower=1, upper=50)))
    * df_full["visibility_score"]
    * (df_full["avg_position"] > 0).astype(int)
)
df_full["depth_gap_score"] = (1 - percentile_rank(df_full["word_count"])) * df_full["visibility_score"]

# Combined baseline score
df_full["baseline_refresh_score"] = (
    0.40 * df_full["visibility_score"]
    + 0.30 * df_full["freshness_risk_score"]
    + 0.25 * df_full["position_opportunity_score"]
    + 0.05 * df_full["depth_gap_score"]
).clip(0, 1)

# Assign reasons and actions
def assign_reason_and_action(row):
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_visible_page", "refresh"
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        return "low_ctr_visible_page", "refresh_and_review_ctr"
    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        return "thin_visible_page", "expand_and_refresh"
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        return "page_one_decay_risk", "refresh"
    return "general_refresh_review", "monitor"

reasons_actions = df_full.apply(assign_reason_and_action, axis=1)
df_full["reason_code"] = [ra[0] for ra in reasons_actions]
df_full["suggested_action"] = [ra[1] for ra in reasons_actions]

# Sort and Rank
df_full = df_full.sort_values(by="baseline_refresh_score", ascending=False)
df_full["baseline_rank"] = range(1, len(df_full) + 1)

# Write output CSV
output_cols = [
    "content_id", "client_id", "baseline_rank", "baseline_refresh_score",
    "reason_code", "suggested_action", "impressions_90d", "clicks_90d", 
    "ctr", "avg_position", "days_since_last_update", "word_count", "trend_direction"
]

output_path = "../outputs/baseline_action_score.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_full[output_cols].to_csv(output_path, index=False)
print(f"Saved baseline action queue to: {output_path}")

# Calculate Precision@50 on the top-50 of this queue against the proxy target label
top_50 = df_full.head(50)
precision_50 = top_50['is_declining_label'].mean()
print(f"Baseline Precision@50 on full dataset: {precision_50:.3f}")


Saved baseline action queue to: ../outputs/baseline_action_score.csv
Baseline Precision@50 on full dataset: 0.360


## 3. Top-20 review

We manually audit the top 10 rows from our generated queue to see if our prioritization logic holds up, naming what action we recommend and under what conditions that recommendation would be incorrect.

### Manual Audit Table

| Rank | Content ID | Action | Reason | Why it's here | What makes it wrong? (Caveat) |
|---|---|---|---|---|---|
| 1 | `content_a5dbb404bdc2` | `refresh_and_review_ctr` | `low_ctr_visible_page` | Massive visibility (79,035 impressions) but a near-zero CTR of 0.07% at position 8.7. | If search results for this term are dominated by Google's Answer Boxes (no-click search) or competitor ad listings. |
| 2 | `content_6ac3ab740bbf` | `refresh_and_review_ctr` | `low_ctr_visible_page` | High visibility (22,462 impressions), very low CTR (0.14%) at rank 4.6, and is actively declining (`down` trend). | If user search intent has fundamentally shifted (e.g. users now want a quick video instead of an article). |
| 3 | `content_69fad7e6c50c` | `monitor` | `general_refresh_review` | High visibility (28,000 impressions), decent CTR (1.32%) at rank 4.7, but in decline. | If the decline is purely seasonal (e.g. tax advice traffic dropping in summer), meaning a refresh won't stop the calendar-driven drop. |
| 4 | `content_03d2673b2553` | `monitor` | `general_refresh_review` | Huge visibility (143,314 impressions), strong position (1.9), stable trend. | If editing a page that currently ranks #1.9 risks disrupting its stable position (don't break what is already working). |
| 5 | `content_399f4eed93b9` | `refresh_and_review_ctr` | `low_ctr_visible_page` | High visibility (127,658 impressions), low CTR (0.43%) at position 3.4. | If the primary search query has a very high rate of local map packs or shopping listings drawing clicks away. |
| 6 | `content_e6e15ac13287` | `refresh_and_review_ctr` | `low_ctr_visible_page` | High visibility (128,704 impressions), low CTR (0.40%) at average position 4.9. | If the page is a navigational hub or directory where users search, find the link, but prefer other page variants. |
| 7 | `content_3ad3a781fa91` | `refresh` | `page_one_decay_risk` | High visibility (145,044 impressions), position on page one (5.2), high age (104 days since update). | If the page content is evergreen (e.g. historical definitions) and does not benefit from regular updates. |
| 8 | `content_fac19fcdfb85` | `refresh_and_review_ctr` | `low_ctr_visible_page` | High visibility (126,611 impressions), very low CTR (0.25%) at position 5.7. | If the search snippet shows meta descriptions that are truncated or missing calls-to-action (a meta-tag fix is needed, not a content rewrite). |
| 9 | `content_681d93f6924d` | `refresh_and_review_ctr` | `low_ctr_visible_page` | Moderate visibility (42,310 impressions), low CTR (0.13%) at position 3.5. | If the topic has become obsolete in the market, making any writing hours spent updating it a poor financial investment. |
| 10 | `content_e1501cdeca69` | `refresh_and_review_ctr` | `low_ctr_visible_page` | High visibility (96,467 impressions), low CTR (0.48%) at average position 4.2. | If competitor pages have much fresher "last updated" dates displayed in the SERPs, making click-throughs fail due to search appearance rather than content quality. |

In [3]:
# Displaying the top 10 rows of our generated queue
top_10 = pd.read_csv("../outputs/baseline_action_score.csv").head(10)
top_10[['content_id', 'baseline_rank', 'baseline_refresh_score', 'reason_code', 'suggested_action', 'impressions_90d', 'avg_position', 'ctr', 'trend_direction']]


,content_id,baseline_rank,baseline_refresh_score,reason_code,suggested_action,impressions_90d,avg_position,ctr,trend_direction
0,content_a5dbb404bdc2,1,0.933579,low_ctr_visible_page,refresh_and_review_ctr,79035,8.7,0.07,stable
1,content_6ac3ab740bbf,2,0.928383,low_ctr_visible_page,refresh_and_review_ctr,22462,4.6,0.14,down
2,content_69fad7e6c50c,3,0.926606,general_refresh_review,monitor,28000,4.7,1.32,down
3,content_03d2673b2553,4,0.923023,general_refresh_review,monitor,143314,1.9,0.83,stable
4,content_399f4eed93b9,5,0.922196,low_ctr_visible_page,refresh_and_review_ctr,127658,3.4,0.43,stable
5,content_e6e15ac13287,6,0.920755,low_ctr_visible_page,refresh_and_review_ctr,128704,4.9,0.40,stable
6,content_3ad3a781fa91,7,0.919993,page_one_decay_risk,refresh,145044,5.2,1.08,stable
7,content_fac19fcdfb85,8,0.918992,low_ctr_visible_page,refresh_and_review_ctr,126611,5.7,0.25,stable
8,content_681d93f6924d,9,0.918318,low_ctr_visible_page,refresh_and_review_ctr,42310,3.5,0.13,stable
9,content_e1501cdeca69,10,0.917194,low_ctr_visible_page,refresh_and_review_ctr,96467,4.2,0.48,stable


## 4. Weak picks + leakage check

### Weak Picks in the Top 10
- **Rank 4 (`content_03d2673b2553`)**: This page is ranked very high in our refresh queue due to its massive visibility (143k impressions) and high position (1.9). However, its trend is **stable** and it is already ranking near the top of Google. Submitting this page for a content rewrite is a **weak pick** because editing a healthy, top-ranking page risks triggering a re-indexing drop that breaks its excellent performance.
- **Rank 1 (`content_a5dbb404bdc2`)**: While it has 79k impressions and rank 8.7, it was updated only **106 days ago**. It is not extremely stale. If the low CTR is due to competitor ads, rewriting the page content will not fix the CTR issue, making this recommendation potentially wasteful.

### Leakage Check
We verified that our baseline score uses only trailing-90-day indicators (`impressions_90d`, `avg_position`, `ctr`, `days_since_last_update`, `word_count`) which were recorded at prediction time. We did not include `trend_pct`, `trend_direction`, or `is_declining_label` in the scoring formula. Our baseline score is **leak-free** and honestly representative of what can be built at decision time.

In [4]:
# Let's verify that no future or label-derived variables are in the baseline action queue columns
output_data = pd.read_csv("../outputs/baseline_action_score.csv")
leaky_vars = ['trend_pct', 'is_declining_label']
found_leaks = [var for var in leaky_vars if var in output_data.columns]
print(f"Leaky variables in output columns: {found_leaks} (Expected: None)")


Leaky variables in output columns: [] (Expected: None)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.